In [1]:
import os
import numpy as np
import torch
import sys
sys.path.append("../")
from protein_mpnn_utils import parse_PDB, StructureDatasetPDB, ProteinMPNN, tied_featurize

### Load model

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

checkpoint_path = '/home/ubuntu/ProteinMPNN/ca_model_weights/v_48_020.pt'
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

hidden_dim = 128
num_layers = 3
backbone_noise = 0
ca_only = True


model = ProteinMPNN(ca_only=ca_only, 
                    num_letters=21, 
                    node_features=hidden_dim, 
                    edge_features=hidden_dim, 
                    hidden_dim=hidden_dim, 
                    num_encoder_layers=num_layers, 
                    num_decoder_layers=num_layers, 
                    augment_eps=backbone_noise, 
                    k_neighbors=checkpoint['num_edges'])

model.to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

ProteinMPNN(
  (features): CA_ProteinFeatures(
    (embeddings): PositionalEncodings(
      (linear): Linear(in_features=66, out_features=16, bias=True)
    )
    (node_embedding): Linear(in_features=3, out_features=128, bias=False)
    (edge_embedding): Linear(in_features=167, out_features=128, bias=False)
    (norm_nodes): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (norm_edges): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (W_v): Linear(in_features=128, out_features=128, bias=True)
  (W_e): Linear(in_features=128, out_features=128, bias=True)
  (W_s): Embedding(21, 128)
  (encoder_layers): ModuleList(
    (0-2): 3 x EncLayer(
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplace=False)
      (dropout3): Dropout(p=0.1, inplace=False)
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm3): LayerNorm((128,), eps=1e-05, elementw

### Wrapper function to return encoder embeddings *h_V* and *h_E*

In [4]:
def MPNN_encoder_embeddings(pdb_path, model):
    '''Get ProteinMPNN encoder embedding for an input pdb '''

    pdb_dict_list = parse_PDB(pdb_path, ca_only=ca_only)
    
    dataset = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=200000)
    
    all_chain_list = [item[-1:] for item in list(pdb_dict_list[0]) if item[:9]=='seq_chain'] #['A','B', 'C',...]
    
    designed_chain_list = all_chain_list
    fixed_chain_list = [letter for letter in all_chain_list if letter not in designed_chain_list]
    chain_id_dict = {}
    chain_id_dict[pdb_dict_list[0]['name']]= (designed_chain_list, fixed_chain_list)
    
    batch_clones = [dataset[0]]
    
    outputs = tied_featurize(
            batch_clones, 
            device, 
            chain_id_dict, 
            fixed_position_dict=None,
            omit_AA_dict=None, 
            tied_positions_dict=None, 
            pssm_dict=None, 
            bias_by_res_dict=None,
            ca_only=ca_only
        )
    
       # Unpack outputs
    X, S, mask, lengths, chain_M, chain_encoding_all, chain_list_list, \
    visible_list_list, masked_list_list, masked_chain_length_list_list, \
    chain_M_pos, omit_AA_mask, residue_idx, dihedral_mask, \
    tied_pos_list_of_lists_list, pssm_coef, pssm_bias, pssm_log_odds_all, \
    bias_by_res_all, tied_beta = outputs
    
    randn_1 = torch.randn(chain_M.shape, device=X.device)
    
    with torch.no_grad():
        h_V, h_E = model(X, S, mask, chain_M*chain_M_pos, residue_idx, chain_encoding_all, randn_1, return_encoder_embs=True)

    return h_V, h_E


In [5]:
from pathlib import Path

# Define dir of pdbs
dir_path = Path("/home/ubuntu/synCP/data/d2dk8a1_synCPs")

# List all files
pdbs = [p for p in dir_path.iterdir() if p.is_file()]

# Create dictionary of node encoder embs, edge encoder embs, and full neighborhood aggregated embs
encoder_node_embs = {pdb.stem: None for pdb in pdbs}
encoder_edge_embs = {pdb.stem: None for pdb in pdbs}

In [6]:
for pdb in pdbs:
    h_V, h_E = MPNN_encoder_embeddings(str(pdb),model)
    encoder_node_embs[pdb.stem] = h_V
    encoder_edge_embs[pdb.stem] = h_E

/home/ubuntu/ProteinMPNN/testing/../protein_mpnn_utils.py:799: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at ../aten/src/ATen/native/Cross.cpp:62.)
  n_2 = F.normalize(torch.cross(u_2, u_1), dim=-1)


In [7]:
embs = list(encoder_node_embs.values())

In [8]:
embs

[tensor([[[ 0.4647, -0.1577, -0.1411,  ..., -0.0940,  0.4843,  0.0751],
          [ 0.0851,  0.1126,  0.3224,  ..., -0.1031, -0.3228,  0.0181],
          [ 0.0530, -0.0900, -0.0523,  ..., -0.0678,  0.2205,  0.1098],
          ...,
          [ 0.0698, -0.2857,  0.0925,  ...,  0.1325, -0.0895,  0.1264],
          [ 0.0043,  0.0105, -0.0168,  ...,  0.0629,  0.0439,  0.0294],
          [-0.0150, -0.0646, -0.1260,  ...,  0.1313, -0.0064,  0.3799]]],
        device='cuda:0'),
 tensor([[[ 0.2317,  0.0660, -0.0196,  ..., -0.0877,  0.1806, -0.0048],
          [ 0.0165,  0.0453,  0.3559,  ...,  0.0449,  0.1105, -0.0284],
          [-0.0529, -0.2431,  0.2186,  ..., -0.0957,  0.0870,  0.3161],
          ...,
          [-0.0317, -0.1276, -0.0968,  ...,  0.2100,  0.0874, -0.0577],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000, -0.0000, -0.0000],
          [ 0.2046, -0.0823,  0.0965,  ...,  0.0427,  0.3164,  0.0850]]],
        device='cuda:0'),
 tensor([[[ 0.4339,  0.3217,  0.3215,  ..., -0.003